In [8]:
from pathlib import Path

from openai import OpenAI
import json
import base64
import jinja2
import pandas as pd

from tqdm.notebook import tqdm

In [9]:
client = OpenAI()

# Load prompts from files in the prompts directory
prompts = {path.stem: path.read_text() for path in Path("prompts/").glob("*.prompt")}

In [ ]:
# Function to process prompt
def evaluate_prompt(prompt, json_data, screenshot_base64=None):
    screenshot_text = ""
    if screenshot_base64:
        try:
            # Add padding if needed
            padding = len(screenshot_base64) % 4
            if padding:
                screenshot_base64 += "=" * (4 - padding)

            # Decode base64 to bytes
            screenshot_bytes = base64.b64decode(screenshot_base64)
            # Convert bytes to string assuming UTF-8 encoding
            screenshot_text = screenshot_bytes.decode("utf-8")
        except Exception as e:
            print(f"Error decoding base64 screenshot: {e}")
            screenshot_text = ""

    with open("templates/basic_prompt.jinja") as template_file:
        combined_prompt_template = jinja2.Template(template_file.read())

    # Render the prompt with JSON data and screenshot text
    combined_prompt = combined_prompt_template.render(
        prompt=prompt,
        json_data=json.dumps(json_data, indent=2),
        screenshot_path=screenshot_text,
    )

    with open("templates/assistant_prompt_content.jinja") as template_file:
        assistant_prompt_template = jinja2.Template(template_file.read())
    assistant_prompt_content = assistant_prompt_template.render()

    messages = [
        dict(role="system", content=assistant_prompt_content),
        dict(role="user", content=combined_prompt),
    ]

    response = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
    content = response.choices[0].message.content

    return content

In [13]:
# Load JSON data (replace with your file path)
screenshot_path = "185986732.png"
with open("185986732.json", "r") as file:
    json_data = json.load(file)

responses = []
for prompt_name, prompt in tqdm(prompts.items()):
    response = evaluate_prompt(prompt, json_data, screenshot_path)
    responses.append(response)

  0%|          | 0/9 [00:00<?, ?it/s]

Error decoding base64 screenshot: 'utf-8' codec can't decode byte 0xd7 in position 0: invalid continuation byte
Error decoding base64 screenshot: 'utf-8' codec can't decode byte 0xd7 in position 0: invalid continuation byte
Error decoding base64 screenshot: 'utf-8' codec can't decode byte 0xd7 in position 0: invalid continuation byte
Error decoding base64 screenshot: 'utf-8' codec can't decode byte 0xd7 in position 0: invalid continuation byte
Error decoding base64 screenshot: 'utf-8' codec can't decode byte 0xd7 in position 0: invalid continuation byte
Error decoding base64 screenshot: 'utf-8' codec can't decode byte 0xd7 in position 0: invalid continuation byte
Error decoding base64 screenshot: 'utf-8' codec can't decode byte 0xd7 in position 0: invalid continuation byte
Error decoding base64 screenshot: 'utf-8' codec can't decode byte 0xd7 in position 0: invalid continuation byte
Error decoding base64 screenshot: 'utf-8' codec can't decode byte 0xd7 in position 0: invalid continuati

In [16]:
df = pd.DataFrame(responses, columns=["responses"])
df.head()

,responses
0,**Is it appropriate for two participants?** \...
1,"**Are features like modals, tabs, carousels, d..."
2,1. Is there at least one image on the page?\nS...
3,**Does the design consider color blindness or ...
4,[Are the steps clearly defined and sequential ...
